In [1]:
%load_ext autoreload
%autoreload 2
import scMPRAforge as scm


2025-08-19 12:40:42.318628: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-19 12:40:42.322857: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/code-server/4.91.1/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

In [2]:
import pandas as pd

## Test wald test

In [3]:

#create dask cluster
from dask.distributed import Client, LocalCluster
cluster=LocalCluster(n_workers=10)
client = Client(cluster)

path="/gpfs/gibbs/pi/reilly/tabula_data/shendure"
name="ortho_primordial"

import os
if os.path.isdir(path+"/"+name):
    print("[+] Model found. Loading...")
    primordial=scm.ortho.load(client,path,name)
    shendure=primordial.training_data
    primordial.precompute_wald(client)
    # primordial.save(path, name) 
else:
    print("[+] Model not found. Creating...")

    #load data
    data_root="/gpfs/gibbs/pi/reilly/tabula_data"
    shendure=scm.scMPRA_data.from_tsv(f"{data_root}/shendure/shendure_counts_grouped.txt")
    
    shendure.set_negative_controls(["minP","noP"])
    shendure.set_reference_cell("Pluripotent")
    shendure.ortho_filter()

    primordial=scm.ortho()
    primordial.criss_cross(client=client,
                       dat=shendure)
    primordial.extract_params(client)
    primordial.precompute_wald(client)  
    primordial.save(path,name)

[+] Model found. Loading...


In [4]:

# Wald tests now hit the precomp cache automatically:
test_fn = scm.build_wald_test_fn(primordial, shendure, cre_mode="vs_reference")
runner  = scm.HypothesisTester(test_fn=test_fn, test_type_name="wald")


In [5]:
# by-cell-type: test many CREs in NeuroectodermBrain vs the 'reference' negative control
hs_ct = scm.make_by_celltype_hypotheses(
    comparison_cell_type="NeuroectodermBrain",
    counts=shendure,
    comparison_cres="all",          # or a list like ["CRE1","CRE2",...]
    reference_cre="reference",      # this is how you labeled minP/noP
    meta="emvar_screen"
)

# by-CRE: test CRE123 across all cell types vs the baseline cell type
hs_cre = scm.make_by_cre_hypotheses(
    comparison_cre="all",
    counts=shendure,
    comparison_cell_types="all",    # or a list
    reference_cell_type="reference",   # will default to counts.reference_cell_type if set
    meta="cell_specificity"
)

# all CREs within each cell type, vs the 'reference' negative control
hs_all_ct = scm.make_all_by_celltype_hypotheses(
    counts=shendure,
    reference_cre="reference",
    meta="emvar_screen",
)

# all cell types for each CRE, vs the dataset’s baseline cell type
hs_all_cre = scm.make_all_by_cre_hypotheses(
    counts=shendure,
    reference_cell_type="reference",  # will be normalized to 'reference'
    meta="cell_specificity",
)




In [6]:
hs_all_ct.to_dataframe()

,comparison_CRE,comparison_cell_type,reference_CRE,reference_cell_type,meta
0,Bend5_chr4_8175,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen
1,Cdk5r1_chr11_12559,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen
2,Col1a1_chr11_15322,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen
3,Col1a2_chr6_77,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen
4,Igfbp4_chr11_16711,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen
...,...,...,...,...,...
1446,Txndc12_chr4_7973,reference,reference,reference,emvar_screen
1447,Lama1_chr17_7787,reference,reference,reference,emvar_screen
1448,Epas1_chr17_10064,reference,reference,reference,emvar_screen
1449,Btg1_chr10_9578,reference,reference,reference,emvar_screen


In [7]:
hs_all_cre.to_dataframe()

,comparison_CRE,comparison_cell_type,reference_CRE,reference_cell_type,meta
0,Bend5_chr4_8168,NeuroectodermBrain,Bend5_chr4_8168,reference,cell_specificity
1,Bend5_chr4_8168,ExEndodermParietal,Bend5_chr4_8168,reference,cell_specificity
2,Bend5_chr4_8168,EpiblastPrimitiveStreak,Bend5_chr4_8168,reference,cell_specificity
3,Bend5_chr4_8168,SurfaceEctoderm,Bend5_chr4_8168,reference,cell_specificity
4,Bend5_chr4_8170,EpiblastPrimitiveStreak,Bend5_chr4_8170,reference,cell_specificity
...,...,...,...,...,...
1240,ubcP,NeuroectodermRostral,ubcP,reference,cell_specificity
1241,ubcP,ExEndodermParietal,ubcP,reference,cell_specificity
1242,ubcP,Haematoendothelial,ubcP,reference,cell_specificity
1243,ubcP,Cardiomyocytes,ubcP,reference,cell_specificity


In [8]:
primordial.wald_precomp

In [10]:
wald_by_cre = runner.run(hs_all_cre).to_dataframe()

KeyboardInterrupt: 

In [13]:
wald_by_cre

,comparison_CRE,comparison_cell_type,reference_CRE,reference_cell_type,meta,test_statistic,p_value,fold_change,flattened,test_type,bh_p
0,Bend5_chr4_8168,NeuroectodermBrain,Bend5_chr4_8168,reference,cell_specificity,NaN,NaN,NaN,False,wald,1.0
1,Bend5_chr4_8168,ExEndodermParietal,Bend5_chr4_8168,reference,cell_specificity,NaN,NaN,NaN,False,wald,1.0
2,Bend5_chr4_8168,EpiblastPrimitiveStreak,Bend5_chr4_8168,reference,cell_specificity,NaN,NaN,NaN,False,wald,1.0
3,Bend5_chr4_8168,SurfaceEctoderm,Bend5_chr4_8168,reference,cell_specificity,NaN,NaN,NaN,False,wald,1.0
4,Bend5_chr4_8170,EpiblastPrimitiveStreak,Bend5_chr4_8170,reference,cell_specificity,NaN,NaN,NaN,False,wald,1.0
...,...,...,...,...,...,...,...,...,...,...,...
1240,ubcP,NeuroectodermRostral,ubcP,reference,cell_specificity,NaN,NaN,NaN,False,wald,1.0
1241,ubcP,ExEndodermParietal,ubcP,reference,cell_specificity,NaN,NaN,NaN,False,wald,1.0
1242,ubcP,Haematoendothelial,ubcP,reference,cell_specificity,NaN,NaN,NaN,False,wald,1.0
1243,ubcP,Cardiomyocytes,ubcP,reference,cell_specificity,NaN,NaN,NaN,False,wald,1.0


In [9]:
wald_by_ct  = runner.run(hs_all_ct).to_dataframe()

NameError: name 'find_treatment_index' is not defined

In [ ]:
wald_by_ct

,comparison_CRE,comparison_cell_type,reference_CRE,reference_cell_type,meta,test_statistic,p_value,fold_change,flattened,test_type,bh_p
0,Bend5_chr4_8175,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen,NaN,NaN,NaN,False,wald,1.0
1,Cdk5r1_chr11_12559,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen,NaN,NaN,NaN,False,wald,1.0
2,Col1a1_chr11_15322,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen,NaN,NaN,NaN,False,wald,1.0
3,Col1a2_chr6_77,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen,NaN,NaN,NaN,False,wald,1.0
4,Igfbp4_chr11_16711,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen,NaN,NaN,NaN,False,wald,1.0
...,...,...,...,...,...,...,...,...,...,...,...
1446,Txndc12_chr4_7973,reference,reference,reference,emvar_screen,NaN,NaN,NaN,False,wald,1.0
1447,Lama1_chr17_7787,reference,reference,reference,emvar_screen,NaN,NaN,NaN,False,wald,1.0
1448,Epas1_chr17_10064,reference,reference,reference,emvar_screen,NaN,NaN,NaN,False,wald,1.0
1449,Btg1_chr10_9578,reference,reference,reference,emvar_screen,NaN,NaN,NaN,False,wald,1.0


In [13]:
some_ct = next(iter(primordial.by_cell_type.model.keys()))
Xnames = primordial.by_cell_type_design[some_ct].result()["nb_regressors"].columns.tolist()
print(Xnames[:10])

['Intercept', "C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8168]", "C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8170]", "C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8172]", "C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8174]", "C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8175]", "C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8179]", "C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8192]", "C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8199]", "C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8201]"]


In [14]:
cluster.close()

2025-08-19 12:40:32,882 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:36993' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'lambda-532d0d01b625f57ae52cfe2f49679de6', 'lambda-f10b535da16a2a151714280ae3fefb66', 'lambda-7c8481f38b3032e121075e2fb91d844c', 'lambda-8861e79eb441911e1b82b2d2aa6119a3', 'lambda-dd52296b100e960ea76d8eeb4f76f73c', 'lambda-4a55c47f83c10d04d7c41ba0415e9d04', 'lambda-830504c375a36ea9c44503674fa21137', 'lambda-454aa038ef716d78d8d450f84a408b9b', 'lambda-dbdbe5e02b26871416e795b9f78bb63e', 'lambda-7b0a049a3c5f5d731404bc063c00fc7a', 'lambda-d7ecc73e205068488d52b8fbc05fecf8', 'lambda-b8c5024887c478be42d23ac425211fbf', 'lambda-a44157b3efc6b2abdd669d9135ddfb25', 'lambda-e411483872318e4a5829f9375bb9616c', 'lambda-775cf473419947618511c8e44db76060', 'lambda-bf9a100a69860aa7f556958e132d2262', 'lambda-78ed592bc0bf37ed366c1e347977c2df', 'lambda-fb59cad9b222952282eb0e299253df3c', 'lambda-08de87c0c6baf321cc4cada71d8b